# Step02: do harmonization
## read in the smoothed FC maps of each tumor masks, then use comBat to harmonize
## save out an averaged brain image file (.gii file)
###
## Xiao Chen
## Wrapped up by Xiao Chen 261119
## chenxiaophd@gmail.com

In [ ]:
# initialization
from neuroHarmonize import harmonizationLearn
from sklearn.decomposition import PCA
from nilearn import datasets, plotting
from nilearn.datasets import fetch_surf_fsaverage
import nibabel as nib
import numpy as np
import pandas as pd
from scipy.io import loadmat
import seaborn as sns
import matplotlib.pyplot as plt
import glob
import os

work_dir = 'xxxxxx'

## read in the selected subject list
## read in the lesion subject list

In [ ]:
# DIRECT subject list
# this is a .mat file listing all the information regarding the DIRECT II dataset
mat = loadmat('xxxxxx/Sub_Info.mat')
subject_list = mat['SubID'].squeeze()
subject_list = [str(s[0]) if isinstance(s, (list, tuple)) else str(s) for s in subject_list]
subject_list = [s.strip("[]'") for s in subject_list]

# build covariates for comBat
Dx = mat['Dx'].squeeze()
Age = mat['Age'].squeeze()
Sex = mat['Sex'].squeeze()
Edu = mat['Edu'].squeeze()
Site = mat['Site'].squeeze()

covars = pd.DataFrame({
    'SITE': Site.ravel(),
    'Dx':   np.asarray(Dx).ravel(),
    'Age':  np.asarray(Age).ravel(),
    'Sex':  np.asarray(Sex).ravel(),
    'Edu':  np.asarray(Edu).ravel()
})
covars['SITE'] = covars['SITE'].astype('category')

# for hemisphere loop
hemi_list = ['LH', 'RH']
group_list = ['MDD', 'HC']

# tumor subject list
df_tumor = pd.read_excel(os.path.join(work_dir, 'data_raw/scale_data_raw_250326.xlsx'))
subject_list_tumor = df_tumor.loc[1:, 'Subject_ID'].tolist()
rum_score_tumor = df_tumor.loc[1:, 'Rumination'].tolist()


## Looping through all tumor subjects to do comBat
## and then save the mean FC maps out
## modified to use the mind flower data

In [ ]:
# modify here to store global or no global signal regression results
output_dir = os.path.join(work_dir, 'xxxxx/mean_FC_mapsS')
os.makedirs(output_dir, exist_ok=True)

# modify here to change the dataset
subject_list_roi = subject_list_tumor

for hemi_label in hemi_list:
    for subj_roi in subject_list_roi:
        data_dir = os.path.join(work_dir, 
                # modify this line to determine the GSR or no GSR data
                # no global
                f"analyses/run_260526/stimuli_FC_SUDMEX_TMS" # change this
                f"/raw_FC_stimuliS/FunSurf{hemi_label}/"
                f"FC_SeedSurfLHSurfRHVolu_FunSurfWCF_SUDMEX_TMS_{subj_roi}_Grey_Matter.nii"
                # # global
                # f"analyses/run_260526/stimuli_FC_SUDMEX_TMS" # change this
                # f"/raw_FC_stimuliS/FunSurf{hemi_label}"
                # f"/FC_SeedSurfLHSurfRHVolu_FunSurfWglobalCF_SUDMEX_TMS_{subj_roi}_Grey_Matter.nii"
                               )
        n_subjects = len(subject_list)
        first_file = os.path.join(data_dir, f'szFC_{subject_list[0]}.func.gii')
        n_vertices = nib.load(first_file).darrays[0].data.shape[0]
        data_matrix = np.zeros((n_subjects, n_vertices))
        missing_subjects = []
        for i, subj in enumerate(subject_list):
            fpath = os.path.join(data_dir, f'szFC_{subj}.func.gii')
            if os.path.exists(fpath):
                gii = nib.load(fpath)
                data_matrix[i, :] = gii.darrays[0].data
            else:
                missing_subjects.append(subj)
                print(f'⚠️ Missing file for subject: {subj}')
        # print('Data matrix shape:', data_matrix.shape)
    
        assert data_matrix.shape[0] == len(Site) == len(Dx) == len(Age) == len(Sex) == len(Edu), "样本数不一致"
        data_matrix = np.asarray(data_matrix, dtype=np.float64)
    
    
        mask_gii = nib.load(f"/dpabi_dir"
                            f"/DPABISurf/SurfTemplates/fsaverage5_{hemi_label.lower()}_cortex.label.gii")
        mask_data = mask_gii.darrays[0].data  
        mask_bool = mask_data.astype(bool)  
        
        assert data_matrix.shape[1] == len(mask_bool), "mask not right!"
        
        # only keep data within the mask
        data_matrix_masked = data_matrix[:, mask_bool]
    
        # !!!!!actually do harmonization
        model, harmonized_data = harmonizationLearn(
            data_matrix_masked,        # n_subjects × n_features
            covars
        )

        # save out MDD patients
        mask = (Dx == 1)
        MDD_data = harmonized_data[mask, :]  
        MDD_mean = np.mean(MDD_data, axis=0)

        n_vertices = len(mask_bool)
        full_data = np.full(n_vertices, np.nan, dtype=np.float32)
        full_data[mask_bool] = MDD_mean.astype(np.float32)

        gii_img = nib.gifti.GiftiImage(darrays=[nib.gifti.GiftiDataArray(full_data)])

        # output data, could be modified
        out_gii = os.path.join(output_dir, f'MDD_group_mean_{str(subj_roi).lower()}_{hemi_label}.func.gii')
        nib.save(gii_img, out_gii)

        # save out HC control
        mask = (Dx == -1)
        HC_data = harmonized_data[mask, :]  
        HC_mean = np.mean(HC_data, axis=0)

        n_vertices = len(mask_bool)
        full_data = np.full(n_vertices, np.nan, dtype=np.float32)
        full_data[mask_bool] = HC_mean.astype(np.float32)

        gii_img = nib.gifti.GiftiImage(darrays=[nib.gifti.GiftiDataArray(full_data)])

        # output data, could be modified
        out_gii = os.path.join(output_dir, f'HC_group_mean_{str(subj_roi).lower()}_{hemi_label}.func.gii')
        nib.save(gii_img, out_gii)

## do plots
## all the mean FC_maps

In [ ]:
output_dir = os.path.join(work_dir, 'xxxxx/mean_FC_mapsS')
result_dir = output_dir   
hemis = hemi_list

views = ["lateral", "medial"]
cmap = "jet"
symmetric = True
dpi = 200

# using fsaverage 5
fsavg = fetch_surf_fsaverage(mesh="fsaverage5")
mesh_left, mesh_right = fsavg.infl_left, fsavg.infl_right
sulc_left, sulc_right = fsavg.sulc_left, fsavg.sulc_right
# ==========================

def load_gii_data(path):
    return np.asarray(nib.load(path).darrays[0].data, dtype=float)

def plot_two_hemis_shared(qname, lh_data, rh_data, out_png):
    if symmetric:
        vmax = np.nanmax(np.abs(np.hstack([lh_data, rh_data])))
        vmin = -vmax
    else:
        vmin = np.nanmin([lh_data.min(), rh_data.min()])
        vmax = np.nanmax([lh_data.max(), rh_data.max()])

    fig = plt.figure(figsize=(10, 8), dpi=dpi)
    gs = fig.add_gridspec(len(views), 3, width_ratios=[1, 1, 0.06], wspace=0.05)

    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    sm = plt.cm.ScalarMappable(norm=norm, cmap=plt.get_cmap(cmap))

    for i, view in enumerate(views):
        # LH
        ax_l = fig.add_subplot(gs[i, 0], projection="3d")
        plotting.plot_surf_stat_map(
            mesh_left, lh_data, bg_map=sulc_left,
            hemi="left", view=view, cmap=cmap,
            vmin=vmin, vmax=vmax, axes=ax_l, colorbar=False
        )
        # RH
        ax_r = fig.add_subplot(gs[i, 1], projection="3d")
        plotting.plot_surf_stat_map(
            mesh_right, rh_data, bg_map=sulc_right,
            hemi="right", view=view, cmap=cmap,
            vmin=vmin, vmax=vmax, axes=ax_r, colorbar=False
        )

    cax = fig.add_subplot(gs[:, 2])
    cbar = plt.colorbar(sm, cax=cax)
    cbar.set_label(qname)

    fig.suptitle(qname, fontsize=12)
    fig.savefig(out_png, bbox_inches="tight", dpi=dpi)
    plt.close(fig)

# ========== main loop ==========
os.makedirs(os.path.join(result_dir, "fig_combined"), exist_ok=True)

for group in group_list:
    for subj in subject_list_roi:
        lh_path = os.path.join(result_dir, f"{group}_group_mean_{subj.lower()}_LH.func.gii")
        rh_path = os.path.join(result_dir, f"{group}_group_mean_{subj.lower()}_RH.func.gii")

        if not (os.path.exists(lh_path) and os.path.exists(rh_path)):
            print(f"missing {group}-{subj} files")
            continue

        lh_data = load_gii_data(lh_path)
        rh_data = load_gii_data(rh_path)

        out_png = os.path.join(result_dir, "fig_combined", f"{group}_group_mean_{subj}_both.png")
        plot_two_hemis_shared(f"{group}-{subj}", lh_data, rh_data, out_png)
        print(f"✅ saved: {out_png}")

In [ ]:
fsavg = datasets.fetch_surf_fsaverage('fsaverage5')

# lh_metric_path = os.path.join(output_dir, f'MDD_group_mean_{subj_roi.lower()}_LH.func.gii')
# rh_metric_path = os.path.join(output_dir, f'MDD_group_mean_{subj_roi.lower()}_RH.func.gii')

lh_metric_path = os.path.join(output_dir, 'rum_causal_mapping_lh.func.gii')
rh_metric_path = os.path.join(output_dir, 'rum_causal_mapping_rh.func.gii')

lh_data = nib.load(lh_metric_path).darrays[0].data
rh_data = nib.load(rh_metric_path).darrays[0].data

In [ ]:
# 交互式显示左半球
lh_disp = lh_data.copy()
lh_disp = np.nan_to_num(lh_disp, nan=0.0)

vmin, vmax = np.nanpercentile(lh_data, [2, 98])
vmax = max(abs(vmin), abs(vmax)); vmin = -vmax  # 对称更直观

view_l = plotting.view_surf(
    fsavg.infl_left,
    lh_disp,
    bg_map=fsavg.sulc_left,
    cmap='cold_white_hot',
    symmetric_cmap=True,
    vmin=vmin, vmax=vmax,      # 👈 关键：固定可见范围
    colorbar=True, 
    bg_on_data=True            # 背景褶皱叠加在数据上
)
view_l

In [ ]:
# 处理 NaN，设置对称色条范围，避免“只有皮层没有颜色”的情况
rh_disp = np.nan_to_num(rh_data, nan=0.0)
vlim = np.nanpercentile(np.abs(rh_disp), 98)
if not np.isfinite(vlim) or vlim <= 0:
    vlim = max(1e-6, float(np.nanmax(np.abs(rh_disp)) or 1.0))

view_r = plotting.view_surf(
    fsavg.infl_right,           # 右半球膨胀表面
    rh_disp,                    # 右半球数据
    bg_map=fsavg.sulc_right,    # 褶皱背景
    cmap='cold_white_hot',
    symmetric_cmap=True,
    vmin=-vlim, vmax=vlim,      # 关键：设定可见范围
    bg_on_data=True,
    colorbar=True
)
view_r  # 在 JupyterLab 中直接显示（或：view_r.open_in_browser()）

## legacy: test for the quality of comBAT

In [ ]:
mean_before = data_matrix_masked.mean(axis=1)
mean_after = harmonized_data.mean(axis=1)

df_before = pd.DataFrame({'MeanValue': mean_before, 'SITE': covars['SITE'], 'Type': 'Before'})
df_after = pd.DataFrame({'MeanValue': mean_after, 'SITE': covars['SITE'], 'Type': 'After'})

df_all = pd.concat([df_before, df_after], axis=0)

plt.figure(figsize=(12,6))
sns.violinplot(data=df_all, x='SITE', y='MeanValue', hue='Type', split=True)
plt.title('Site effect before vs after ComBat (mean feature per subject)')
plt.tight_layout()
plt.show()

In [ ]:
pca = PCA(n_components=2)
coords_b = pca.fit_transform(data_matrix_masked)
coords_a = pca.fit_transform(harmonized_data)

df_b = pd.DataFrame({'PC1': coords_b[:,0], 'PC2': coords_b[:,1], 'Site': covars['SITE'], 'Type': 'Before'})
df_a = pd.DataFrame({'PC1': coords_a[:,0], 'PC2': coords_a[:,1], 'Site': covars['SITE'], 'Type': 'After'})
df = pd.concat([df_b, df_a])

g = sns.FacetGrid(df, col='Type', hue='Site', height=5)
g.map_dataframe(sns.scatterplot, x='PC1', y='PC2')
g.add_legend()
plt.show()